# Freesound SVM classification (core)

This notebook is organized into four stages:

1. Load imports and helper functions
2. Download/prepare the training and test data
3. Train and evaluate the SVM model
4. Download/predict the query sound and save the results

The plotting is handled separately in the visualization notebook.

In [20]:
import json
import os
import pickle
import shutil
from pathlib import Path

import freesound as fs
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.svm import SVC

In [21]:
descriptors = [
    'lowlevel.spectral_centroid.mean',
    'lowlevel.spectral_contrast.mean',
    'lowlevel.dissonance.mean',
    'lowlevel.hfc.mean',
    'lowlevel.mfcc.mean',
    'sfx.logattacktime.mean',
    'sfx.inharmonicity.mean',
]
descriptorMapping = {
    0: 'lowlevel.spectral_centroid.mean',
    1: 'lowlevel.dissonance.mean',
    2: 'lowlevel.hfc.mean',
    3: 'sfx.logattacktime.mean',
    4: 'sfx.inharmonicity.mean',
    5: 'lowlevel.spectral_contrast.mean.0',
    6: 'lowlevel.spectral_contrast.mean.1',
    7: 'lowlevel.spectral_contrast.mean.2',
    8: 'lowlevel.spectral_contrast.mean.3',
    9: 'lowlevel.spectral_contrast.mean.4',
    10: 'lowlevel.spectral_contrast.mean.5',
    11: 'lowlevel.mfcc.mean.0',
    12: 'lowlevel.mfcc.mean.1',
    13: 'lowlevel.mfcc.mean.2',
    14: 'lowlevel.mfcc.mean.3',
    15: 'lowlevel.mfcc.mean.4',
    16: 'lowlevel.mfcc.mean.5'
}

def convFtrDict2List(ftrDict):
    ftr = []
    for key in range(len(descriptorMapping.keys())):
        try:
            ftrName = '.'.join(descriptorMapping[key].split('.')[:-1])
            ind = int(descriptorMapping[key].split('.')[-1])
            ftr.append(ftrDict[ftrName][0][ind])
        except Exception:
            ftr.append(ftrDict[descriptorMapping[key]][0])
    return np.array(ftr, dtype=float)

def fetchDataDetails(inputDir, descExt='.json'):
    dataDetails = {}
    for path in Path(inputDir).rglob(f'*{descExt}'):
        sound_dir = path.parent
        class_dir = sound_dir.parent
        cname = class_dir.name
        sname = sound_dir.name
        if cname not in dataDetails:
            dataDetails[cname] = {}
        with open(path, 'r', encoding='utf-8') as file_obj:
            fDict = json.load(file_obj)
        dataDetails[cname][sname] = {'file': path.name, 'feature': fDict}
    return dataDetails

def download_sounds_freesound(
    queryText="",
    tag=None,
    duration=None,
    API_Key="",
    outputDir="",
    topNResults=5,
    featureExt='.json',
):
    """Download Freesound previews and descriptor JSON files for a query."""

    if queryText == "":
        print("\nProvide a query text to search for sounds")
        return -1

    if API_Key == "":
        print("\nYou need a valid Freesound API key to download sounds.")
        print("Please apply for one here: www.freesound.org/apiv2/apply/\n")
        return -1

    if outputDir == "":
        print("\nPlease provide a valid output directory.")
        return -1

    output_dir = Path(outputDir)
    output_dir.mkdir(parents=True, exist_ok=True)

    fsClnt = fs.FreesoundClient()
    fsClnt.set_token(API_Key, "token")

    flt_dur = f" duration:[{duration[0]} TO {duration[1]}]" if isinstance(duration, tuple) else ""
    flt_tag = f"tag:{tag}" if isinstance(tag, str) and tag != "" else ""
    search_filter = f"{flt_tag}{flt_dur}"

    page_size = 30
    if search_filter != "":
        qRes = fsClnt.text_search(
            query=queryText,
            filter=search_filter,
            sort="score",
            fields="id,name,previews,username,url,analysis",
            descriptors=','.join(descriptors),
            page_size=page_size,
            normalized=1,
        )
    else:
        qRes = fsClnt.text_search(
            query=queryText,
            sort="score",
            fields="id,name,previews,username,url,analysis",
            descriptors=','.join(descriptors),
            page_size=page_size,
            normalized=1,
        )

    query_dir = output_dir / queryText
    if query_dir.exists():
        shutil.rmtree(query_dir)
    query_dir.mkdir(parents=True, exist_ok=True)

    pageNo = 1
    sndCnt = 0
    indCnt = 0
    totalSnds = min(qRes.count, 200)
    downloadedSounds = []

    while True:
        if indCnt >= totalSnds:
            print("Not able to download the required number of sounds.")
            break

        sound = qRes[indCnt - ((pageNo - 1) * page_size)]
        sound_dir = query_dir / str(sound.id)
        if sound_dir.exists():
            shutil.rmtree(sound_dir)
        sound_dir.mkdir(parents=True, exist_ok=True)

        preview_name = str(sound.previews.preview_lq_mp3.split("/")[-1])
        mp3Path = sound_dir / preview_name
        ftrPath = Path(str(mp3Path).replace('.mp3', featureExt))

        print(f"Downloading mp3 preview and descriptors for sound with id: {sound.id}")

        try:
            fs.FSRequest.retrieve(sound.previews.preview_lq_mp3, fsClnt, str(mp3Path))
            features = {}
            for desc in descriptors:
                features[desc] = [eval("sound.analysis." + desc)]
            with open(ftrPath, 'w', encoding='utf-8') as file_obj:
                json.dump(features, file_obj)
            sndCnt += 1
            downloadedSounds.append([str(sound.id), sound.url])
        except Exception:
            if sound_dir.exists():
                shutil.rmtree(sound_dir)

        indCnt += 1
        if indCnt % page_size == 0:
            qRes = qRes.next_page()
            pageNo += 1

        if sndCnt >= topNResults:
            break

    with open(query_dir / f"{queryText}_SoundList.txt", 'w', encoding='utf-8') as fid:
        for elem in downloadedSounds:
            fid.write('\t'.join(elem) + '\n')

def build_svm_dataset(input_dir: str, desc_input: list[int]):
    if len(desc_input) == 0:
        raise ValueError('desc_input must contain at least one descriptor index')

    data_details = fetchDataDetails(input_dir)
    features = []
    labels = []
    sound_ids = []

    for class_name in sorted(data_details.keys()):
        for sound_name in sorted(data_details[class_name].keys()):
            full_vector = convFtrDict2List(data_details[class_name][sound_name]['feature'])
            features.append(full_vector[desc_input])
            labels.append(class_name)
            sound_ids.append(sound_name)

    return np.array(features, dtype=float), np.array(labels), sound_ids

def _standardize_train_query(train_data: np.ndarray, query_data: np.ndarray | None = None):
    mean = np.mean(train_data, axis=0, keepdims=True)
    std = np.std(train_data, axis=0, keepdims=True)
    std[std == 0] = 1.0
    scaled_train = (train_data - mean) / std
    if query_data is None:
        return scaled_train, mean, std
    scaled_query = (query_data - mean.ravel()) / std.ravel()
    return scaled_train, scaled_query, mean, std

def _infer_query_class(query_file: str) -> str | None:
    query_path = Path(query_file)
    if len(query_path.parents) >= 2:
        return query_path.parents[1].name
    return None

## Data preparation

Prepare the dataset used by the SVM.

This stage can automatically download the training sounds from Freesound if the local descriptor files are missing, and then creates the train/test split.

In [22]:
# ===== CONTROL PARAMETERS =====
training_dir = 'svm_train_sounds'
query_sound_name = 'viola'
selected_descriptors = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
test_size = 0.2
random_state = 42
kernel = 'rbf'
C = 10.0
gamma = 'scale'
results_file = 'svm_results/svm_core_results.pkl'

# Training download controls
training_queries = [
    ('violin', 'pizzicato', 20),
    ('naobo', '', 20),
    ('trumpet', 'single-note', 20),
]
training_duration = (0, 3)

# Query download controls (used in the query cell below)
query_top_n_results = 1
query_duration = (0, 3)
query_tag = 'pizzicato'
# =============================

if 'build_svm_dataset' not in globals() or 'download_sounds_freesound' not in globals():
    raise NameError(
        'Required helper functions are not loaded. Run Cell 3 (helper functions) before running this cell.'
    )

full_features, full_labels, sound_ids = build_svm_dataset(training_dir, selected_descriptors)

if len(full_features) == 0:
    print('No training descriptors found locally. Searching in Freesound to build the training set...')
    load_dotenv()
    api_key = os.environ.get('FREESOUND_API_KEY')
    if not api_key:
        raise EnvironmentError(
            'FREESOUND_API_KEY not set. '
            'Create a .env file with FREESOUND_API_KEY=your_key (see .env.example).'
        )

    for class_name, class_tag, top_n_results in training_queries:
        download_sounds_freesound(
            queryText=class_name,
            tag=class_tag,
            duration=training_duration,
            API_Key=api_key,
            outputDir=training_dir,
            topNResults=top_n_results,
        )

    full_features, full_labels, sound_ids = build_svm_dataset(training_dir, selected_descriptors)

if len(full_features) == 0:
    raise FileNotFoundError(
        'No training descriptor files were found or downloaded under svm_train_sounds. '
        'Check your API key and Freesound connectivity, then rerun this cell.'
    )

train_indices, test_indices = train_test_split(
    np.arange(len(full_features)),
    test_size=test_size,
    stratify=full_labels,
    random_state=random_state,
)

train_features = full_features[train_indices]
train_labels = full_labels[train_indices]
test_features = full_features[test_indices]
test_labels = full_labels[test_indices]

print(f'Total dataset: {len(full_features)} samples')
print(f'Training set prepared: {len(train_features)} samples')
print(f'Test set prepared: {len(test_features)} samples')

Total dataset: 60 samples
Training set prepared: 48 samples
Test set prepared: 12 samples


## Model training

Train the SVM using the prepared training split and standardize features before fitting the classifier.

In [23]:
# Train the SVM model using the prepared training split
scaled_train_features, mean, std = _standardize_train_query(train_features)
scaled_test_features = (test_features - mean.ravel()) / std.ravel()

model = SVC(kernel=kernel, C=C, gamma=gamma, probability=True)
model.fit(scaled_train_features, train_labels)

print(f'Model trained with kernel={kernel}, C={C}, gamma={gamma}')

Model trained with kernel=rbf, C=10.0, gamma=scale


## Evaluation

Evaluate the trained SVM on the training and test splits using accuracy, precision, recall, F1-score, and the confusion matrix.

In [24]:
# Evaluate trained model on training and test sets
train_pred = model.predict(scaled_train_features)
test_pred = model.predict(scaled_test_features)

print(f'Train accuracy: {accuracy_score(train_labels, train_pred) * 100:.1f}%')
print(f'Test accuracy: {accuracy_score(test_labels, test_pred) * 100:.1f}%')
print(f'Test precision: {precision_score(test_labels, test_pred, average="weighted", zero_division=0) * 100:.1f}%')
print(f'Test recall: {recall_score(test_labels, test_pred, average="weighted", zero_division=0) * 100:.1f}%')
print(f'Test F1: {f1_score(test_labels, test_pred, average="weighted", zero_division=0) * 100:.1f}%')

classes = sorted(np.unique(test_labels))
print('Confusion matrix (test):')
print(confusion_matrix(test_labels, test_pred, labels=classes))

Train accuracy: 100.0%
Test accuracy: 100.0%
Test precision: 100.0%
Test recall: 100.0%
Test F1: 100.0%
Confusion matrix (test):
[[4 0 0]
 [0 4 0]
 [0 0 4]]


## Query prediction and result export

Download the query sound if needed, classify it with the trained model, and save the result bundle for the visualization notebook.

In [25]:
# Resolve or download query descriptor, then predict and save results bundle
query_root = Path('svm_query_sound') / query_sound_name
query_candidates = sorted(query_root.rglob('*.json'))

if not query_candidates:
    print(f"No local query descriptor found for '{query_sound_name}'. Searching in Freesound...")
    load_dotenv()
    api_key = os.environ.get('FREESOUND_API_KEY')
    if not api_key:
        raise EnvironmentError(
            'FREESOUND_API_KEY not set. '
            'Create a .env file with FREESOUND_API_KEY=your_key (see .env.example).'
        )

    download_sounds_freesound(
        queryText=query_sound_name,
        tag=query_tag,
        duration=query_duration,
        API_Key=api_key,
        outputDir='svm_query_sound',
        topNResults=query_top_n_results,
    )
    query_candidates = sorted(query_root.rglob('*.json'))

if not query_candidates:
    available = []
    base = Path('svm_query_sound')
    if base.exists():
        available = sorted([p.name for p in base.iterdir() if p.is_dir()])
    raise FileNotFoundError(
        f"No query descriptor found for '{query_sound_name}'.\n"
        f"Expected under: {query_root}\n"
        f"Available query folders: {available}\n"
        f"Freesound search returned no usable descriptor files."
    )

query_file = str(query_candidates[0])
print(f'Using query descriptor: {query_file}')

with open(query_file, 'r', encoding='utf-8') as file_obj:
    query_dict = json.load(file_obj)
query_features = convFtrDict2List(query_dict)[selected_descriptors].astype(float)
scaled_query_features = (query_features - mean.ravel()) / std.ravel()

predicted_label = model.predict([scaled_query_features])[0]
probabilities = model.predict_proba([scaled_query_features])[0]
predicted_probability = float(dict(zip(model.classes_, probabilities))[predicted_label])
query_label = _infer_query_class(query_file)

combined_scaled_features = np.vstack((scaled_train_features, scaled_test_features))
combined_labels = np.concatenate((train_labels, test_labels))
combined_sound_ids = [sound_ids[i] for i in train_indices] + [sound_ids[i] for i in test_indices]
test_mask = np.concatenate((np.zeros(len(train_features), dtype=bool), np.ones(len(test_features), dtype=bool)))

bundle = {
    'combined_scaled_features': combined_scaled_features,
    'combined_labels': combined_labels,
    'combined_sound_ids': combined_sound_ids,
    'test_mask': test_mask,
    'support_indices': model.support_,
    'scaled_query_features': scaled_query_features,
    'query_label': query_label,
    'predicted_label': predicted_label,
    'predicted_probability': predicted_probability,
    'kernel': kernel,
    'C': C,
    'gamma': gamma,
}

results_path = Path(results_file)
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, 'wb') as file_obj:
    pickle.dump(bundle, file_obj)

print(f'Predicted query class: {predicted_label} ({predicted_probability * 100:.1f}%)')
print(f'Saved core results to: {results_path}')

Using query descriptor: svm_query_sound/viola/374382/374382_2475994-lq.json
Predicted query class: violin (93.9%)
Saved core results to: svm_results/svm_core_results.pkl
